In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, RandomOverSampler
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier, BaseEnsemble
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from tensorflow.keras.callbacks import ReduceLROnPlateau

acc = []

data = pd.read_csv("train.csv")

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["TotalCharges"] = data["TotalCharges"].fillna(0)
data['TotalCharges'] = data['TotalCharges'].astype(int)
data["tenure"] = data["tenure"].fillna(data["MonthlyCharges"])

data["Months avg Payment"] = data['TotalCharges'] / data["tenure"]
data["Months payed"] = data['TotalCharges'] / data["MonthlyCharges"].astype(int)
data["Months avg Payment"] = data["Months avg Payment"].fillna(0)
data["Is Automatic"] = data["PaymentMethod"].isin([ 'Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)
data["SeniorWithDependents"] = ((data["SeniorCitizen"] == 1) & (data["Dependents"] == "Yes")).astype(int)
data["HasFamily"] = ((data["Partner"] == "Yes") & (data["Dependents"] == "Yes")).astype(int)

data["IsNoStableClient"] = (
    (
        (data["tenure"] < 10).astype(int) +
        (data["TechSupport"] == "No").astype(int) +
        (data["OnlineSecurity"] == "No").astype(int) +
        (data["StreamingMovies"] == "No").astype(int) +
        (data["StreamingTV"] == "No").astype(int) +
        (data["Months avg Payment"] > 90).astype(int) +
        (data["Months payed"] < 20).astype(int) +
        (data["Is Automatic"] == 1).astype(int) +
        (data["SeniorWithDependents"] == 1).astype(int) +
        (data["PaymentMethod"] == "Electronic check").astype(int)
    ) >= 5
).astype(int)


X = pd.get_dummies(data.drop(["Churn", "id"], axis=1))
y = data['Churn'].apply(lambda x:1 if x=="Yes" else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

smote_tomek = SMOTE(random_state=70)
X_smoted, y_smoted = smote_tomek.fit_resample(X_train, y_train)

for i in range(6):
  model = keras.Sequential([
    layers.Dense(2, activation = "swish", input_dim=X_train.shape[1]),
    layers.Dense(20, activation = "swish"),
    layers.Dense(1, activation = "sigmoid")
  ])

  earlystop = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights = True)
  reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.75, patience=10, min_lr=1e-6)
  model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

  model.fit(X_train, y_train, epochs=100, batch_size=40, validation_split=0.1, callbacks=[earlystop, reduce_lr], class_weight={0:1.1, 1:1.6})

  #model = BaggingClassifier(n_estimators=300, random_state=100)
  #model.fit(X_smoted, y_smoted)

  y_prediction = model.predict(X_test)
  y_prediction = [1 if val >=0.5 else 0 for val in y_prediction]

  report = classification_report(y_test, y_prediction, output_dict=True)
  accuracy = report["accuracy"]
  print(report)
  acc.append(accuracy)

print(acc)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6623 - loss: 5.7932 - val_accuracy: 0.7126 - val_loss: 0.5850 - learning_rate: 0.0010
Epoch 2/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7105 - loss: 0.7679 - val_accuracy: 0.7067 - val_loss: 0.6332 - learning_rate: 0.0010
Epoch 3/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6581 - loss: 0.8611 - val_accuracy: 0.7205 - val_loss: 0.5711 - learning_rate: 0.0010
Epoch 4/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7208 - loss: 0.7639 - val_accuracy: 0.7283 - val_loss: 0.5800 - learning_rate: 0.0010
Epoch 5/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7221 - loss: 0.7971 - val_accuracy: 0.5118 - val_loss: 0.7055 - learning_rate: 0.0010
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6423 - loss: 0.8715 - val_accuracy: 0.7165 - val_loss: 0.6602 - learning_rate: 0.0010
Epoch 7/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7191 - loss: 0.7895 - val_a

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2610 - loss: 88.9622 - val_accuracy: 0.2933 - val_loss: 4.4130 - learning_rate: 0.0010
Epoch 2/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5326 - loss: 1.9340 - val_accuracy: 0.7480 - val_loss: 0.5632 - learning_rate: 0.0010
Epoch 3/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7349 - loss: 0.7769 - val_accuracy: 0.7480 - val_loss: 0.5606 - learning_rate: 0.0010
Epoch 4/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7339 - loss: 0.7728 - val_accuracy: 0.7480 - val_loss: 0.5648 - learning_rate: 0.0010
Epoch 5/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7345 - loss: 0.7653 - val_accuracy: 0.7480 - val_loss: 0.5623 - learning_rate: 0.0010
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7409 - loss: 0.7567 - val_accuracy: 0.7520 - val_loss: 0.5830 - learning_rate: 0.0010
Epoch 7/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7300 - loss: 0.7777 - val_

...

115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7191 - loss: 31.1773 - val_accuracy: 0.4193 - val_loss: 0.6990 - learning_rate: 0.0010
Epoch 2/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7090 - loss: 0.7202 - val_accuracy: 0.7441 - val_loss: 0.5180 - learning_rate: 0.0010
Epoch 3/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7447 - loss: 0.7343 - val_accuracy: 0.7657 - val_loss: 0.4985 - learning_rate: 0.0010
Epoch 4/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7376 - loss: 0.7075 - val_accuracy: 0.7598 - val_loss: 0.4818 - learning_rate: 0.0010
Epoch 5/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7273 - loss: 0.7063 - val_accuracy: 0.7618 - val_loss: 0.5240 - learning_rate: 0.0010
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7542 - loss: 0.7144 - val_accuracy: 0.7657 - val_loss: 0.4948 - learning_rate: 0.0010
Epoch 7/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7365 - loss: 0.7221 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7385 - loss: 0.8671 - val_accuracy: 0.7480 - val_loss: 0.6303 - learning_rate: 0.0010
Epoch 2/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7308 - loss: 0.8065 - val_accuracy: 0.7480 - val_loss: 0.5935 - learning_rate: 0.0010
Epoch 3/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7358 - loss: 0.7879 - val_accuracy: 0.7480 - val_loss: 0.5818 - learning_rate: 0.0010
Epoch 4/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7375 - loss: 0.7809 - val_accuracy: 0.7480 - val_loss: 0.5771 - learning_rate: 0.0010
Epoch 5/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7365 - loss: 0.7737 - val_accuracy: 0.7480 - val_loss: 0.5728 - learning_rate: 0.0010
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7363 - loss: 0.7729 - val_accuracy: 0.7677 - val_loss: 0.5678 - learning_rate: 0.0010
Epoch 7/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7620 - loss: 0.7623 - val_a

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4373 - loss: 7.6292 - val_accuracy: 0.6713 - val_loss: 0.5919 - learning_rate: 0.0010
Epoch 2/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6890 - loss: 0.7598 - val_accuracy: 0.6909 - val_loss: 0.5845 - learning_rate: 0.0010
Epoch 3/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6945 - loss: 0.7808 - val_accuracy: 0.7224 - val_loss: 0.5721 - learning_rate: 0.0010
Epoch 4/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7262 - loss: 0.7568 - val_accuracy: 0.7303 - val_loss: 0.5697 - learning_rate: 0.0010
Epoch 5/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7145 - loss: 0.7650 - val_accuracy: 0.7323 - val_loss: 0.5830 - learning_rate: 0.0010
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7194 - loss: 0.7750 - val_accuracy: 0.7323 - val_loss: 0.5656 - learning_rate: 0.0010
Epoch 7/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7445 - loss: 0.7481 - val_a